# Multimodal: Images + Text (SigLIP 2 and Qwen3-VL)

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/06-Vision-Language.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
!pip install -q -U transformers
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_siglip_zeroshot.py', 'q_vlm_json.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
from urllib.request import urlopen
device = "cuda" if torch.cuda.is_available() else "cpu"
def show(img, width=300):
    display(img.resize((width, int(width * img.height / img.width))))

## SigLIP 2: putting images and text in the same vector space

Trained on a large number of image + caption pairs so that matching images and texts get nearby vectors (contrastive learning).
Applications: text-to-image search, zero-shot classification, and serving as the vision encoder of a multimodal LLM.

In [ ]:
from transformers import AutoModel, AutoProcessor
siglip_id = "google/siglip2-base-patch16-224"
siglip = AutoModel.from_pretrained(siglip_id, dtype=torch.float16).to(device).eval()
siglip_proc = AutoProcessor.from_pretrained(siglip_id)

### Data: the first 300 test images of Oxford-IIIT Pet

We use streaming mode, so there's no need to download the whole dataset.

In [ ]:
from datasets import load_dataset
pets_stream = load_dataset("timm/oxford-iiit-pet", split="test", streaming=True)
class_names = pets_stream.features["label"].names
samples = list(pets_stream.shuffle(seed=0, buffer_size=1000).take(300))
images = [s["image"].convert("RGB") for s in samples]
labels = np.array([s["label"] for s in samples])
len(images)

In [ ]:
@torch.no_grad()
def image_embeddings(images, batch_size=64):
    embs = []
    for i in range(0, len(images), batch_size):
        inputs = siglip_proc(images=images[i:i + batch_size], return_tensors="pt").to(device)
        out = siglip.get_image_features(**inputs)
        out = out if isinstance(out, torch.Tensor) else out.pooler_output
        embs.append(out.float())
    return torch.nn.functional.normalize(torch.cat(embs), dim=-1)

@torch.no_grad()
def text_embeddings(texts):
    # SigLIP 2 was trained with text padded to 64 tokens, so we must do the same at inference
    inputs = siglip_proc(text=texts, padding="max_length", max_length=64, return_tensors="pt").to(device)
    out = siglip.get_text_features(**inputs)
    out = out if isinstance(out, torch.Tensor) else out.pooler_output
    return torch.nn.functional.normalize(out.float(), dim=-1)

img_emb = image_embeddings(images)
img_emb.shape

### Text-to-image search

In [ ]:
def search(query, k=5):
    scores = (text_embeddings([query]) @ img_emb.T)[0]
    top = scores.topk(k).indices.tolist()
    display(Image.fromarray(np.concatenate([np.array(images[i].resize((160, 160))) for i in top], axis=1)))
search("a black cat")

In [ ]:
search("a dog lying on the grass")

In [ ]:
search("a dog wearing clothes")

### Image-to-image search

In [ ]:
q = 7
show(images[q], 160)
scores = img_emb @ img_emb[q]
top = scores.topk(6).indices.tolist()[1:]
display(Image.fromarray(np.concatenate([np.array(images[i].resize((160, 160))) for i in top], axis=1)))

## Q: Zero-shot classification
Build a text prompt from each of the 37 breed names (e.g. `"a photo of a {name}, a type of pet."`) and compute the zero-shot accuracy on these 300 images.

Compare it with the result from the previous unit, which used supervised training on 3680 images.

In [ ]:
# %load q_siglip_zeroshot.py
%run -i q_siglip_zeroshot.py

## Vision Language Model (VLM): Qwen3-VL

VLM = vision encoder + LLM. The image is cut into small patches, turned into "visual tokens", and fed into the LLM together with the text tokens.

[Qwen3-VL-2B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct) takes about 4.5GB in fp16, so it runs on a T4.

In [ ]:
import gc
del siglip; gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModelForImageTextToText
vlm_id = "Qwen/Qwen3-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(vlm_id)
vlm = AutoModelForImageTextToText.from_pretrained(vlm_id, dtype=torch.float16, device_map="auto")

In [ ]:
def ask(images, question, max_new_tokens=256):
    images = images if isinstance(images, list) else [images]
    messages = [{"role": "user", "content": [{"type": "image", "image": im} for im in images] +
                                            [{"type": "text", "text": question}]}]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(vlm.device)
    out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [ ]:
img = Image.open(urlopen("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg")).convert("RGB")
show(img)
print(ask(img, "Describe this image in detail."))

In [ ]:
print(ask(img, "What breed is the dog in the image? What is it doing?"))

In [ ]:
show(images[0], 200)
print(ask(images[0], "What animal is this? What breed? Answer in one sentence."))
print("Answer:", class_names[labels[0]])

### Text recognition (OCR) and information extraction

Let's draw our own receipt to test it.

In [ ]:
receipt = Image.new("RGB", (520, 400), "white")
draw = ImageDraw.Draw(receipt)
font = ImageFont.load_default(size=26)
lines = ["SUNSHINE CAFE", "2026-11-10  14:32", "", "Latte        x2   $9.00", "Bagel        x1   $3.50",
         "Cheesecake   x1   $5.25", "", "Subtotal         $17.75", "Tax 8%            $1.42", "TOTAL            $19.17"]
for i, line in enumerate(lines):
    draw.text((30, 20 + i * 36), line, fill="black", font=font)
show(receipt, 350)

In [ ]:
print(ask(receipt, "Which store is this receipt from? What is the total? What items were bought?"))

### Multiple images

In [ ]:
print(ask([images[1], images[2]], "What do the animals in these two images have in common, and how do they differ?"))

## Q
Have the VLM output the receipt as JSON (store name, date, list of items with quantity and price, total amount) and parse it with `json.loads`.
Then check in code whether the item prices add up to the Subtotal.

In [ ]:
# %load q_vlm_json.py
%run -i q_vlm_json.py